# Tutorial 5: Cloud Integration with AWS and GCP

## What You Will Learn

- Configure AWS Fargate and GCP targets in a manifest
- Use the artifact store for cloud storage (S3, GCS)
- Estimate costs with dry-run planning
- Structure multi-cloud manifests

## Prerequisites

- Completed Tutorials 1-2
- `pip install scalable[cloud]`
- AWS/GCP credentials (or follow along conceptually)

> **Note:** This notebook demonstrates configuration and planning. Actual cloud deployment requires valid credentials and infrastructure.

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-cloud-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: AWS Target Configuration

The AWS provider uses `dask-cloudprovider` to launch Dask workers on Fargate or EC2.

In [ ]:
aws_manifest = """\
version: 1
project:
  name: energy-model-aws
  default_storage: s3://my-bucket/scalable-runs/

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

  aws:
    provider: aws
    region: us-east-1
    cluster_type: fargate
    worker_cpu: 4096
    worker_mem: 16384
    image: 123456789.dkr.ecr.us-east-1.amazonaws.com/energy-model:latest
    execution_role_arn: arn:aws:iam::123456789:role/ecsTaskExecutionRole
    task_role_arn: arn:aws:iam::123456789:role/scalableTaskRole
    subnets:
      - subnet-abc123
      - subnet-def456
    security_groups:
      - sg-xyz789
    adaptive:
      minimum: 2
      maximum: 20

components:
  gcam:
    image: 123456789.dkr.ecr.us-east-1.amazonaws.com/gcam:7.0
    cpus: 4
    memory: 16G
    tags: [multi-sector-dynamics, energy]

  postprocess:
    cpus: 2
    memory: 8G
    tags: [analysis]

tasks:
  run_gcam:
    component: gcam
    cache: true
    outputs:
      database: dir

  aggregate:
    component: postprocess
    cache: true
"""

with open("scalable.yaml", "w") as f:
    f.write(aws_manifest)

print("AWS manifest written.")
print("\nKey configuration:")
print("  cluster_type: fargate (serverless containers)")
print("  worker_cpu: 4096 (= 4 vCPU)")
print("  worker_mem: 16384 (= 16 GiB)")
print("  adaptive: min=2, max=20 workers")

### Fargate CPU/Memory Configurations

| CPU (units) | Memory (MiB) | Use Case |
|-------------|--------------|----------|
| 1024 | 4096 | Light tasks, I/O-bound |
| 4096 | 16384 | Standard compute |
| 16384 | 65536 | Memory-intensive models |

## Step 2: Validate and Plan (Local Target)

We can validate and plan against the local target without AWS credentials.

In [ ]:
from scalable import ScalableSession

# Use the local target for validation
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
report = session.validate()

print(f"Manifest valid: {report.ok}")
if not report.ok:
    for e in report.errors:
        print(f"  ERROR: {e.message}")

In [ ]:
# Plan for local execution
plan = session.plan(dry_run=True)
print(f"Target: {plan.target_name}")
print(f"Provider: {plan.provider_name}")
print(f"Scale plan: {plan.scale_plan}")
print(f"Manifest lock: {plan.manifest_lock}")

## Step 3: Cost Estimation

Scalable includes cost tables for cloud providers. Let's explore the cost estimation module.

In [ ]:
from scalable.costing import CostEstimate

# CostEstimate is included in dry-run plan output for cloud targets
print("CostEstimate fields:")
print(f"  - total: Total estimated cost")
print(f"  - compute: Fargate/EC2 compute cost")
print(f"  - storage: S3/GCS storage cost")
print(f"  - transfer: Data transfer cost")

# Check if cost estimate is available
if hasattr(plan, 'cost_estimate') and plan.cost_estimate:
    print(f"\nEstimated cost: ${plan.cost_estimate.total:.2f}")
else:
    print("\nCost estimation available for cloud targets (aws, gcp)")
    print("Run: scalable run ./scalable.yaml --target aws --dry-run")

## Step 4: Artifact Store

The artifact store provides unified storage across local and cloud backends.

In [ ]:
from scalable.artifacts import build_artifact_store

# Local artifact store (always available)
local_store = build_artifact_store("./artifacts")
print(f"Local store: {local_store}")
print(f"Store type: {type(local_store).__name__}")

# Create a sample output file
os.makedirs("output", exist_ok=True)
with open("output/results.csv", "w") as f:
    f.write("scenario,demand_mw\n1,35.2\n2,28.7\n")

# Store an artifact
ref = local_store.put("output/results.csv", "runs/demo/results.csv")
print(f"\nStored artifact: {ref}")

In [ ]:
# Retrieve the artifact
retrieved_path = local_store.get("runs/demo/results.csv", "./downloads/results.csv")
print(f"Retrieved to: {retrieved_path}")

with open(retrieved_path) as f:
    print(f"Contents: {f.read()}")

### Cloud Storage (S3/GCS)

With `scalable[cloud]` installed, the same API works with remote URIs:

```python
# S3
s3_store = build_artifact_store("s3://my-bucket/artifacts/")
ref = s3_store.put("local/output.csv", "runs/run-001/output.csv")

# GCS
gcs_store = build_artifact_store("gs://my-bucket/artifacts/")
ref = gcs_store.put("local/output.csv", "runs/run-001/output.csv")
```

The store auto-detects the URI scheme and uses `fsspec` backends (`s3fs`, `gcsfs`).

## Step 5: GCP Target Configuration

For reference, here's a GCP/GKE manifest:

In [ ]:
gcp_manifest = """\
version: 1
project:
  name: energy-model-gke
  default_storage: gs://my-bucket/scalable-runs/

targets:
  gke:
    provider: kubernetes
    namespace: energy-prod
    image: gcr.io/my-project/energy-model:latest
    adaptive:
      minimum: 2
      maximum: 20

components:
  gcam:
    image: gcr.io/my-project/gcam:7.0
    cpus: 8
    memory: 32G
    tags: [multi-sector-dynamics, energy]
    env:
      GCAM_DATA: /data/gcam

  postprocess:
    image: gcr.io/my-project/postprocess:latest
    cpus: 4
    memory: 16G

tasks:
  run_gcam:
    component: gcam
    cache: true
  aggregate:
    component: postprocess
    cache: true
"""

print("GCP/GKE manifest structure:")
print(gcp_manifest)

## Step 6: Environment Variable Template

Production deployments use environment variables for credentials and configuration.

In [ ]:
env_template = """\
# .env.cloud (do NOT commit secrets)
AWS_REGION=us-east-1
S3_BUCKET=energy-prod-artifacts
ECR_IMAGE=123456789.dkr.ecr.us-east-1.amazonaws.com/energy-model:latest
EXECUTION_ROLE_ARN=arn:aws:iam::123456789:role/ecsTaskExecutionRole
TASK_ROLE_ARN=arn:aws:iam::123456789:role/scalableTaskRole
SUBNET_A=subnet-abc123
SUBNET_B=subnet-def456
SG_ID=sg-xyz789
SCALABLE_CACHE_REMOTE=s3://energy-prod-artifacts/cache/
"""

print("Environment variable template for cloud deployment:")
print(env_template)
print("Usage: set -a && source .env.cloud && set +a")

## Step 7: Cloud + Cache Integration

Combine cloud execution with remote caching so repeated runs across different machines share results:

```bash
export SCALABLE_CACHE_REMOTE=s3://my-bucket/scalable-cache/
```

Cache lookup order:
1. Local disk (fast, per-machine)
2. Remote store (slower, shared across team)
3. Execute function (slowest, produces new entry)

In [ ]:
# Demonstrate the settings that control remote caching
from scalable.common import settings

print("Cache-related settings:")
print(f"  Local cache dir: {settings.cache_dir}")
print(f"  Remote cache URI: {settings.cache_remote_uri}")
print(f"  Default storage: {settings.default_storage}")
print(f"\nTo enable remote cache:")
print(f"  export SCALABLE_CACHE_REMOTE=s3://bucket/cache/")

## Summary

1. AWS Fargate provides serverless Dask workers with auto-scaling
2. GCP/GKE uses Kubernetes provider with container-native execution
3. Artifact store (`build_artifact_store`) works identically for local/S3/GCS
4. Cost estimation is built into dry-run planning
5. Environment variables keep credentials out of manifests
6. Remote caching enables cross-machine result sharing

## Next Steps

- **Tutorial 6**: Monitor cloud costs and performance through telemetry
- **Tutorial 8**: Full Kubernetes deployment walkthrough
- **Tutorial 7**: Handle cloud-specific transient failures

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")